# Topic: Exponents and Logarithms - Log space scaling, probability underflows, and the Log-Sum-Exp stabilization trick
 
## 1. CORE PROPERTIES & MATH
- **Exponent Function**: $y = b^x$. Grows extremely rapidly.
- **Logarithm Function**: $y = \log_b(x)$. The inverse of exponents. Slow growth.
- **Core rules**:
  - Product Rule: $\log(xy) = \log(x) + \log(y)$
  - Quotient Rule: $\log(x/y) = \log(x) - \log(y)$
  - Power Rule: $\log(x^k) = k \log(x)$
 
## 2. LOG SPACE SCALING: PREVENTING PROBABILITY UNDERFLOWS
- In machine learning (e.g. Naive Bayes, HMMs), we compute joint likelihoods by multiplying many independent probabilities:
  $$P(D) = P(x_1) \times P(x_2) \times \dots \times P(x_N)$$
- **The Underflow Problem**: Since $P(x_i) \in [0, 1]$, multiplying hundreds of small decimal numbers generates a product so small that it underflows to `0.0` in 64-bit float representation.
- **The Solution (Log Space)**: Take the logarithm of both sides, converting products to sums:
  $$\log P(D) = \log P(x_1) + \log P(x_2) + \dots + \log P(x_N)$$
  This operates in negative values (e.g. $-300.5$), which are safely within IEEE 754 float ranges.
 
## 3. THE LOG-SUM-EXP TRICK
- In Softmax normalization or partition functions, we compute:
  $$y = \log \sum_{i=1}^N e^{x_i}$$
- **Numerical Overflow/Underflow**: If $x_i$ are very large (e.g. $1000$), $e^{1000}$ overflows to `inf`. If they are very small negative numbers (e.g. $-1000$), they underflow to `0.0`.
- **The Stabilization Math**:
  - Let $c = \max_i x_i$. Subtract $c$ inside the exponent and add it back outside the log:
    $$\log \sum_{i=1}^N e^{x_i} = \log \sum_{i=1}^N e^{x_i - c + c} = \log \left( e^c \sum_{i=1}^N e^{x_i - c} \right) = c + \log \sum_{i=1}^N e^{x_i - c}$$
  - Since the maximum exponent term is now $x_i - c = 0$, $e^0 = 1$, the sum is guaranteed to be at least $1.0$, preventing underflow, and the remaining terms are negative, preventing overflow.
 
---


In [ ]:
import math
import numpy as np

# 1. Simulating Probability Underflow
print("--- Joint Probability Underflow ---")
probs = [0.01] * 200  # Multiplying 0.01 two hundred times

# Naive Multiplication
product = 1.0
for p in probs:
    product *= p
print(f"Naive joint product: {product}")  # Expected: 0.0 (Underflow!)

# Log Space Addition
log_sum = 0.0
for p in probs:
    log_sum += math.log(p)
print(f"Log space sum:       {log_sum:.4f}")  # Expected: -921.0340 (Stable representation!)



In [ ]:
# 2. Log-Sum-Exp Trick Implementation
def naive_log_sum_exp(x_array):
    """Calculates log(sum(exp(x_i))) directly. Subject to overflows."""
    return np.log(np.sum(np.exp(x_array)))

def stabilized_log_sum_exp(x_array):
    """Calculates log(sum(exp(x_i))) using Log-Sum-Exp stabilization."""
    c = np.max(x_array)
    return c + np.log(np.sum(np.exp(x_array - c)))

print("\n--- Log-Sum-Exp Trick Validation ---")
# Large inputs that trigger overflow
large_inputs = np.array([1000.0, 1001.0, 1002.0])

try:
    print(f"Naive result:      {naive_log_sum_exp(large_inputs)}")  # Expected: inf
except RuntimeWarning:
    pass

print(f"Stabilized result: {stabilized_log_sum_exp(large_inputs):.4f}")
# Expected output: 1002.4076 (Correctly computed!)
